In [ ]:
# Step 1. Set Google Colab Connection
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 2. Import Python Packages: Pandas, Numpy, Statistics, and Scipy
#   Optimization via Scipy

import pandas as pd
import numpy as np
import statistics as st
import scipy
from scipy import stats
import scipy.optimize as sci
from scipy.optimize import minimize
from scipy.optimize import fsolve

In [ ]:
# Step 3. I/O Folders and Filenames

# I/O Folders
InputPath="/content/drive/MyDrive/KRG/"
OutputPath="/content/drive/MyDrive/KRG/"

# Inputfile: MI Datafile
MIDataFile="MIData-KRG.csv"

# OutputFile: MI Parmeters
MIResultsFile="MIParameters.csv"


In [4]:
# Step 4. Open File and Define Variables
MIData = pd.read_csv(InputPath+MIDataFile)
Cost = MIData['Cost']
Size = MIData['Size']
Volatility = MIData['Volatility']
POV = MIData['POV']
X = MIData[['Size','Volatility','POV']]
n,k = X.shape
  # n=number of observations
  # k=number of input variables

print(MIData)


             Date    Size  Volatility     POV        Cost
0       8/14/2020  0.0927    0.498829  0.0303   -0.357895
1       8/14/2020  0.0568    0.344376  0.0333  171.617139
2       8/14/2020  0.0534    0.193020  0.0415  -26.072693
3       8/14/2020  0.0580    0.471967  0.0478  260.844290
4       8/14/2020  0.1500    0.337130  0.0538  126.922787
...           ...     ...         ...     ...         ...
233130  6/30/2022  0.1768    0.525394  0.5871  197.113837
233131  6/30/2022  0.3853    0.322805  0.5872  124.846741
233132  6/30/2022  0.2236    0.261822  0.5887   95.185164
233133  6/30/2022  0.1311    0.132569  0.5944   62.545765
233134  6/30/2022  0.7262    0.457231  0.5979  154.009555

[233135 rows x 5 columns]


In [5]:
# Step 5. Definte MI Cost Optimization function
#   We are solving via non-linear least squares
#   Data is compiled from Investor/Broker data
#   This is our Optimization Objective Function, "f"

def MI_NonLinearOLS(initial_guess, Size, POV, Volatility, Cost):
    a1, a2, a3, a4, b1 = initial_guess
    EstCost=(a1*Size**a2*Volatility**a3)*(b1*POV**a4+(1-b1))
    return np.sum(np.square(Cost - EstCost))


In [6]:
# Step 6. Run Non-Linear Regression using Non-Linear Least Squares
#         To estimate the MI Parameters

# Starting Values for MI Parameter Values
a1 = 1000
a2 = 0.5
a3 = 0.5
a4 = 0.5
b1 = 0.90
initial_guess = [a1, a2, a3, a4, b1]

# Set Upper and Lower Bounds
MyBounds = [(0, 2000), (0, 10), (0, 10), (0, 10), (0, 1),]

# Run Non-Linear Regression using scipy minimize function
results = minimize(MI_NonLinearOLS, initial_guess, bounds = MyBounds, args = (Size, POV, Volatility, Cost))

# Display Non-Linear Regression Results
print('Non-Linear Regression Results:')
print(results)

# MI Parameter Values
a1 = results.x[0]
a2 = results.x[1]
a3 = results.x[2]
a4 = results.x[3]
b1 = results.x[4]

# Display MI Parameters
print(' ')
print('MI Parameters:')
print(a1, a2, a3, a4, b1)

Non-Linear Regression Results:
  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 4945541246.699385
        x: [ 8.823e+02  3.539e-01  7.553e-01  8.238e-01  9.645e-01]
      nit: 40
      jac: [ 0.000e+00 -6.571e+04 -2.546e+05 -5.484e+04  5.894e+04]
     nfev: 318
     njev: 53
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>
 
MI Parameters:
882.2838099679868 0.35394979940907545 0.7552580955176748 0.8237515033443212 0.9644582577771039


In [7]:
# Step 7. Calculate Model Error
#    Model error is not provided in our non-linear regression
#    We need the mean square error (MSE) and the standard error (SE)

# Actual Cost
# Cost

# Calculate EstCost from IStar Model Error
EstCost=(a1*Size**a2*Volatility**a3)*(b1*POV**a4+(1-b1))

# Calculate MSE
MSE = np.mean((Cost - EstCost) ** 2)
SE = MSE**0.5

# Display Model Error Results to Screen
print(MSE, SE)

21213.207998367405 145.6475471759391


In [8]:
# Step 8. Save Results

DataResults = [[a1,a2,a3,a4,b1,MSE,SE]]
MIResults = pd.DataFrame(DataResults, columns=['a1','a2','a3','a4','b1','MSE','SE'])
MIResults.to_csv(OutputPath+MIResultsFile, index = False, header=True)
MIResults.head()


,a1,a2,a3,a4,b1,MSE,SE
0,882.28381,0.35395,0.755258,0.823752,0.964458,21213.207998,145.647547
